#  Pauli Encoding is the Solution!

This study and code are presented by the team `eQoSystem` as part of Q-Volution Rigetti's Challenge!






## 1. Loading Graph &  Pauli Correlation Encoding

We set the qubits to be 12, and the compression rate to be 2, to fit our Dataset B with 180 nodes

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations
from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile


print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  #
num_qubits = 12                 
k = 2                           

# ==========================================
# PAULI-CORRELATION ENCODING (PCE)
# ==========================================


list_size = num_nodes // 3 #To encode the same number of nodes into each set:
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)


## 2. Ansatz Creation

In [ ]:
ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
estimator.options.default_shots = 10000

experiment_result = []

## 3. Loss Function, Chain rule and Parameter shifting

In [ ]:
# ==========================================
# LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
alpha_loss = 24  
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def evaluate_expectations(theta): #To run a Qiskit estimation job and retrieve expectation values
    
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
   
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    return np.concatenate([E_x, E_y, E_z])

def calc_loss_and_chain_rule(E):
    #Analytically calculates the loss and the gradient dL/dE
    
    
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    # Chain rule 
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    # Chain rule for the Regularization term
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

# ==========================================
# PARAMETER SHIFT RULE 
# ==========================================
def param_shift_gradient(theta):
    
    num_params = len(theta)
    shift = np.pi / 2.0 #The shift step
    
    #Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) 
    
    
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Run estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # extract results 
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        # Transpose if the shape is (240, 60) to make it (60, 240)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    #Compute the jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    #Apply Chain Rule
    E_current = evaluate_expectations(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    grad = dL_dE @ J
    
    # Save
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad

## 4. Adam Optimizer with parameter shift rule

In [ ]:
def adam_optimize_param_shift(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    print(f"Starting optimization with Parameter Shift Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    for i in range(max_iter):
        t += 1
        
        loss, grad = param_shift_gradient(theta_i)
        print(f"--- Adam Step {i+1}/{max_iter} --- | Loss: {loss:.6f}")
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

best_params = adam_optimize_param_shift(
    theta0=initial_params,
    lr=0.1,        
    max_iter=1000
)

print("\nOptimization Complete")
np.save("Parameters_VQE_REPS4_K3", best_params) #To save the parametrs for later

## 5. Decoding the results and Calculate the Cut

In [ ]:
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nRaw Cut Size: {best_cut}")

## 6. Post Processing algorithms:

In [ ]:
# One round Single-bit swap search
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Optimized Cut Size:", best_cut)

## 6.2 Greedy Local Search Post Processing

In [ ]:
cur_bits = np.zeros(num_nodes, dtype=int)
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits[i] = 1
    else:
        cur_bits[i] = 0

# Helper to calculate the full absolute cut size once
def calculate_full_cut(graph, bits):
    cut = 0
    for u, v, data in graph.edges(data=True):
        if bits[u] != bits[v]:
            cut += data.get('weight', 1.0)
    return cut

current_cut = calculate_full_cut(graph, cur_bits)
print(f"\nInitial Quantum Cut (Before Local Search): {current_cut}")

# 2. Iterative 1-Bit and 2-Bit Swap Search
search_iteration = 2
improvement_found = True

while improvement_found:
    improvement_found = False
    search_iteration += 1
    
    # ---1-BIT SWAP SEARCH ---
    
    for u in graph.nodes():
        delta = 0
        # Calculate exactly how the cut changes if we flip node 
        for v in graph.neighbors(u):
            w = graph[u][v].get('weight', 1.0)
            if cur_bits[u] == cur_bits[v]:
                delta += w  # They were in the same partition
            else:
                delta -= w  # They were in different partitions
                
        # If flipping strictly improves the cut, apply the flip
        if delta > 0:
            cur_bits[u] = 1 - cur_bits[u]
            current_cut += delta
            improvement_found = True

    # --- 2-BIT SWAP SEARCH ---
    # If 1-bit swaps get stuck in a local minimum, we try flipping pairs of connected nodes
    if not improvement_found:
        for u, v in graph.edges():
            delta = 0
            
            # Change in cut for neighbors of u (excluding v)
            for n_u in graph.neighbors(u):
                if n_u == v: continue
                w = graph[u][n_u].get('weight', 1.0)
                if cur_bits[u] == cur_bits[n_u]: delta += w
                else: delta -= w
                    
            # Change in cut for neighbors of v (excluding u)
            for n_v in graph.neighbors(v):
                if n_v == u: continue
                w = graph[v][n_v].get('weight', 1.0)
                if cur_bits[v] == cur_bits[n_v]: delta += w
                else: delta -= w
                    
            if delta > 0:
                cur_bits[u] = 1 - cur_bits[u]
                cur_bits[v] = 1 - cur_bits[v]
                current_cut += delta
                improvement_found = True
                break # Break to restart the 1-bit sweep with the new landscape

print(f"Post-processing complete after {search_iteration} iteration.")
print(f"Final Deep-Optimized Cut Size: {current_cut}")

## 7. Translation and run in QVM

In [ ]:
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pyquil.paulis import sI, sX, sY, sZ
from pytket.passes import AutoRebase
from pyquil.api import WavefunctionSimulator
import numpy as np
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pytket.passes import AutoRebase
from pytket.circuit import OpType
from pyquil.paulis import sI, sX, sY, sZ
from pyquil.api import WavefunctionSimulator
import numpy as np


print("\nTranslating Qiskit circuit to PyQuil via PyTKET...")

# 1. Bind the trained parameters to your Qiskit ansatz
best_params_loaded = np.load("Parameters_VQE_REPS4_K3.npy")
bound_circuit = ansatz.assign_parameters(best_params_loaded)

# 2. Convert Qiskit -> TKET -> PyQuil Program
tket_circ = qiskit_to_tk(bound_circuit)
rebase = AutoRebase({OpType.CX, OpType.Rx, OpType.Rz})
rebase.apply(tket_circ)

pyquil_prog = tk_to_pyquil(tket_circ)

# 3. Translate Qiskit SparsePauliOp observables to PyQuil PauliTerms
def qiskit_to_pyquil_pauli(qiskit_op):
    pauli_str = qiskit_op.paulis.to_labels()[0]
    term = sI()  # Start with the Identity multiplier
    
    # Enumerate backwards to match Qiskit's endianness
    for qubit_idx, char in enumerate(reversed(pauli_str)):
        if char == 'X':
            term *= sX(qubit_idx)
        elif char == 'Y':
            term *= sY(qubit_idx)
        elif char == 'Z':
            term *= sZ(qubit_idx)
    return term

pyquil_obs_x = [qiskit_to_pyquil_pauli(op) for op in observables_x]
pyquil_obs_y = [qiskit_to_pyquil_pauli(op) for op in observables_y]
pyquil_obs_z = [qiskit_to_pyquil_pauli(op) for op in observables_z]


#Evaluating exact expectation values on PyQuil QVM WavefunctionSimulator

wfn_sim = WavefunctionSimulator()

# Calculate expectation values by passing the entire list of PauliTerms directly
E_x_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_x))
E_y_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_y))
E_z_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_z))

E_final_qvm = np.concatenate([E_x_qvm, E_y_qvm, E_z_qvm])
print("QVM Evaluation Complete!\n")


# Decode the bits directly from the QVM expectations
cur_bits = []
for i in range(num_nodes):
    if E_final_qvm[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut based on QVM
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Raw Cut Size from PyQuil QVM before Local Search: {best_cut}")

## 7.2 Runing on QVM and preparing for Pauili Twirling Error mitigation

In [ ]:
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pyquil.paulis import sI, sX, sY, sZ
from pytket.passes import AutoRebase
from pyquil.api import WavefunctionSimulator
import numpy as np
from pytket.circuit import OpType
from pyquil.quilbase import Gate
from pyquil import Program, get_qc
from pyquil.gates import H, RX, MEASURE
from pyquil.quil import address_qubits.
from pyquil.quil import address_qubits
from mitiq import pt


print("\nApplying Pauli Twirling and translating to PyQuil...")

ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=2)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
best_params_loaded = np.load("pce_optimal_theta_adam_reps4qbt12Continue2010_linear.npy")
bound_circuit = ansatz.assign_parameters(best_params_loaded)

# --- MITIQ: Generate twirled ensemble ---
num_rc_circuits = 6
print(f"Generating {num_rc_circuits} twirled circuits...")

twirled_qiskit_circuits = pt.generate_pauli_twirl_variants(
    bound_circuit, 
    num_circuits=num_rc_circuits
)

rebase = AutoRebase({OpType.CX, OpType.Rx, OpType.Rz})
pyquil_rc_progs = []

# Translate each twirled circuit to PyQuil
for circ in twirled_qiskit_circuits:
    tket_circ = qiskit_to_tk(circ)
    rebase.apply(tket_circ)
    pyquil_rc_progs.append(tk_to_pyquil(tket_circ))

def qiskit_to_pyquil_pauli(qiskit_op):
    pauli_str = qiskit_op.paulis.to_labels()[0]
    term = sI() 
    for qubit_idx, char in enumerate(reversed(pauli_str)):
        if char == 'X':
            term *= sX(qubit_idx)
        elif char == 'Y':
            term *= sY(qubit_idx)
        elif char == 'Z':
            term *= sZ(qubit_idx)
    return term

print("Translating observables...")
pyquil_obs_x = [qiskit_to_pyquil_pauli(op) for op in observables_x]
pyquil_obs_y = [qiskit_to_pyquil_pauli(op) for op in observables_y]
pyquil_obs_z = [qiskit_to_pyquil_pauli(op) for op in observables_z]


print("Evaluating exact expectation values on PyQuil QVM...")

wfn_sim = WavefunctionSimulator()
E_x_qvm = np.real(wfn_sim.expectation(pyquil_rc_progs[0], pauli_terms=pyquil_obs_x))
E_y_qvm = np.real(wfn_sim.expectation(pyquil_rc_progs[0], pauli_terms=pyquil_obs_y))
E_z_qvm = np.real(wfn_sim.expectation(pyquil_rc_progs[0], pauli_terms=pyquil_obs_z))

E_final_qvm = np.concatenate([E_x_qvm, E_y_qvm, E_z_qvm])

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = [1 if val >= 0 else 0 for val in E_final_qvm]
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()


## 8. Real ANKAA-3 Execution with Pauli Twirling

In [ ]:
from pyquil.quil import Program
from pyquil.gates import H, RX, MEASURE
import numpy as np

print("Preparing twirled ensemble for Rigetti Ankaa-3...")

# Chain mapping
qubit_mapping = {
    0: 2, 
    1: 9, 
    2: 8, 
    3: 7, 
    4: 14, 
    5: 21, 
    6: 22, 
    7: 15, 
    8: 16, 
    9: 23, 
    10: 30, 
    11: 31
}

qc = get_qc("Ankaa-3")
total_shots = 10000
# Distribute shots evenly across the RC ensemble
shots_per_circ = total_shots // num_rc_circuits

def run_mapped_basis(mapped_prog, basis, mapping, qc, shots):
    # FORCE the compiler to use our exact physical qubits, no auto-rerouting!
    prog = Program(f'PRAGMA INITIAL_REWIRE "NAIVE"')
    prog += mapped_prog
    
    # Declare the readout register (ro) sized to our logical circuit
    ro = prog.declare("ro", "BIT", len(mapping))
    
    # Append the basis transformation gates to the physical qubits
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
    # Measure the physical qubits into the logical indices of the readout register
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    exe = qc.compile(prog)
    result = qc.run(exe)
    
    
    if hasattr(result, "readout_data"):
        return result.readout_data.get("ro") # PyQuil 4.x
    else:
        return result.get_register_map().get("ro") # Older PyQuil 3.x

def get_expectations(bitstrings, qiskit_observables, shots):
    # Convert {0, 1} bitstrings to {+1, -1}
    spins = 1 - 2 * bitstrings 
    evs = []
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
        evs.append(np.mean(obs_val))
    return np.array(evs)

#Execute
E_x_qpu_total, E_y_qpu_total, E_z_qpu_total = 0, 0, 0

print(f"\nExecuting {num_rc_circuits} twirled circuits on Ankaa-3 ({shots_per_circ} shots each)...")
for i, prog in enumerate(pyquil_rc_progs):
    print(f"  -> Running circuit {i+1}/{num_rc_circuits}")
    
    # This correctly physically addresses the circuit using our mapping dictionary
    mapped_prog = address_qubits(prog, qubit_mapping)
    
    print("     [1/3] Measuring X basis...")
    res_x = run_mapped_basis(mapped_prog, 'X', qubit_mapping, qc, shots_per_circ)
    print("     [2/3] Measuring Y basis...")
    res_y = run_mapped_basis(mapped_prog, 'Y', qubit_mapping, qc, shots_per_circ)
    print("     [3/3] Measuring Z basis...")
    res_z = run_mapped_basis(mapped_prog, 'Z', qubit_mapping, qc, shots_per_circ)
    
    E_x_qpu_total += get_expectations(res_x, observables_x, shots_per_circ)
    E_y_qpu_total += get_expectations(res_y, observables_y, shots_per_circ)
    E_z_qpu_total += get_expectations(res_z, observables_z, shots_per_circ)

# Average out the expectation values
E_x_qpu = E_x_qpu_total / num_rc_circuits
E_y_qpu = E_y_qpu_total / num_rc_circuits
E_z_qpu = E_z_qpu_total / num_rc_circuits

E_final_qpu = np.concatenate([E_x_qpu, E_y_qpu, E_z_qpu])
print("\nAnkaa-3 QPU Evaluation Complete!\n")

# --- QPU Decoding & Local Search ---
cur_bits = [1 if val >= 0 else 0 for val in E_final_qpu]
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from Ankaa-3 before Local Search: {best_cut}")

## 9. Searching for the best qubits chain to run the circuit based on calibration data

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx

df = pd.read_csv('Ankaa-3_device_specs.csv') 

# Parse nodes
nodes_df = df[df['Qubit'].notna()].copy()
nodes_df['Qubit'] = nodes_df['Qubit'].astype(int)

node_features = {}
for _, row in nodes_df.iterrows():
    q = int(row['Qubit'])
    t1 = row['T1 (µs)'] if pd.notna(row['T1 (µs)']) else 0.1
    t2 = row['T2 (µs)'] if pd.notna(row['T2 (µs)']) else 0.1
    f1q = row['f1QRB'] if pd.notna(row['f1QRB']) else 0.5
    node_features[q] = {'T1': t1, 'T2': t2, 'f1Q': f1q}

# Parse edges
edges_df = df[df['Pair'].notna()].copy()
edges = {}
graph = nx.Graph()

for _, row in edges_df.iterrows():
    pair = row['Pair']
    q1, q2 = map(int, pair.split('-'))
    f_iswap = row['fISWAP'] if pd.notna(row['fISWAP']) else 0.5
    edges[(q1, q2)] = f_iswap
    edges[(q2, q1)] = f_iswap
    graph.add_edge(q1, q2)

def score_chain(chain):
    f1_prod = np.prod([node_features[q]['f1Q'] for q in chain])
    fiswap_prod = np.prod([edges[(chain[i], chain[i+1])] for i in range(len(chain)-1)])
    min_t1 = min([node_features[q]['T1'] for q in chain])
    min_t2 = min([node_features[q]['T2'] for q in chain])
    fidelity_score = f1_prod * fiswap_prod
    composite_score = fidelity_score * min_t1 * min_t2
    return composite_score, fidelity_score, min_t1, min_t2

all_paths = []
def dfs(current_node, current_path):
    if len(current_path) == 9:
        all_paths.append(list(current_path))
        return
    for neighbor in graph.neighbors(current_node):
        if neighbor not in current_path:
            current_path.append(neighbor)
            dfs(neighbor, current_path)
            current_path.pop()

for node in graph.nodes():
    dfs(node, [node])

scored_paths = []
for path in all_paths:
    if path[0] < path[-1]:
        sc, fid, t1, t2 = score_chain(path)
        scored_paths.append({
            'chain': path,
            'score': sc,
            'fidelity_prod': fid,
            'min_T1': t1,
            'min_T2': t2
        })

scored_paths.sort(key=lambda x: x['score'], reverse=True)

print("top_15_mappings_9q = {")
for i in range(15):
    p = scored_paths[i]
    chain_str = ", ".join([f"{idx}: {q}" for idx, q in enumerate(p['chain'])])
    print(f"    \"Rank {i+1}\": {{{chain_str}}}, # Score: {p['score']:.2f}, Fid: {p['fidelity_prod']:.4f}, T1: {p['min_T1']:.2f}, T2: {p['min_T2']:.2f}")
print("}")

---
## Experimenets:
Some runned experiments with few iteration, with warm start from previous trainings on considerable number of iterations


In [3]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations
from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile


print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  
num_qubits = 12                 
k = 2                           


list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)


ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()

estimator.options.default_shots = 10000

experiment_result = []

alpha_loss = 24  
beta_loss = 0.5

v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def evaluate_expectations(theta):
    """Helper to get pure expectation values for a single parameter array."""
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
    # Extract 1D arrays of expectation values
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    return np.concatenate([E_x, E_y, E_z])

def calc_loss_and_chain_rule(E):
    """Analytically calculates the loss and the gradient dL/dE."""
    # Precompute common terms for speed
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        # dL/dE for connected nodes
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE


def param_shift_gradient(theta):
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # FIX: Add a new axis to force a Cartesian product broadcast
    # Now it evaluates all 240 parameter sets against all 60 observables
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        # Transpose if the shape is (240, 60) to make it (60, 240)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad


warm_start_file = "Optimal_Reps4_k2.npy"
num_new_params = ansatz.num_parameters 

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)


def adam_optimize_param_shift(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    print(f"Starting optimization with Parameter Shift Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    for i in range(max_iter):
        t += 1
        
        loss, grad = param_shift_gradient(theta_i)
        print(f"--- Adam Step {i+1}/{max_iter} --- | Loss: {loss:.6f}")
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

best_params = adam_optimize_param_shift(
    theta0=initial_params,
    lr=0.1,        
    max_iter=20
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue2010_linear4.npy", best_params)


def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")


for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from Optimal_Reps4_k2.npy...
Starting optimization with Parameter Shift Adam (lr=0.1)...


/tmp/ipykernel_219229/3005752682.py:52: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)


--- Adam Step 1/20 --- | Loss: -5079.877110
--- Adam Step 2/20 --- | Loss: -2131.137542
--- Adam Step 3/20 --- | Loss: -3565.915300
--- Adam Step 4/20 --- | Loss: -4109.418107
--- Adam Step 5/20 --- | Loss: -3903.744178
--- Adam Step 6/20 --- | Loss: -4384.596007
--- Adam Step 7/20 --- | Loss: -4394.408118
--- Adam Step 8/20 --- | Loss: -4347.432302
--- Adam Step 9/20 --- | Loss: -4515.026588
--- Adam Step 10/20 --- | Loss: -4668.815423
--- Adam Step 11/20 --- | Loss: -4630.707595
--- Adam Step 12/20 --- | Loss: -4623.083467
--- Adam Step 13/20 --- | Loss: -4725.522965
--- Adam Step 14/20 --- | Loss: -4820.166722
--- Adam Step 15/20 --- | Loss: -4814.928423
--- Adam Step 16/20 --- | Loss: -4826.551332
--- Adam Step 17/20 --- | Loss: -4854.024026
--- Adam Step 18/20 --- | Loss: -4916.345121
--- Adam Step 19/20 --- | Loss: -4952.219293
--- Adam Step 20/20 --- | Loss: -4955.139011

Adam Optimization Complete!

Initial Cut Size before Local Search: 6561.394409484451
Final Optimized Cut Siz

## GREEDY Search

In [4]:
cur_bits = np.zeros(num_nodes, dtype=int)
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits[i] = 1
    else:
        cur_bits[i] = 0

# Helper to calculate the full absolute cut size once
def calculate_full_cut(graph, bits):
    cut = 0
    for u, v, data in graph.edges(data=True):
        if bits[u] != bits[v]:
            cut += data.get('weight', 1.0)
    return cut

current_cut = calculate_full_cut(graph, cur_bits)
print(f"\nInitial Quantum Cut (Before Local Search): {current_cut}")

# 2. Iterative 1-Bit and 2-Bit Swap Search
search_iteration = 2
improvement_found = True

while improvement_found:
    improvement_found = False
    search_iteration += 1
    
    # ---1-BIT SWAP SEARCH ---
    
    for u in graph.nodes():
        delta = 0
        # Calculate exactly how the cut changes if we flip node 
        for v in graph.neighbors(u):
            w = graph[u][v].get('weight', 1.0)
            if cur_bits[u] == cur_bits[v]:
                delta += w  # They were in the same partition
            else:
                delta -= w  # They were in different partitions
                
        # If flipping strictly improves the cut, apply the flip
        if delta > 0:
            cur_bits[u] = 1 - cur_bits[u]
            current_cut += delta
            improvement_found = True

    # --- 2-BIT SWAP SEARCH ---
    # If 1-bit swaps get stuck in a local minimum, we try flipping pairs of connected nodes
    if not improvement_found:
        for u, v in graph.edges():
            delta = 0
            
            # Change in cut for neighbors of u (excluding v)
            for n_u in graph.neighbors(u):
                if n_u == v: continue
                w = graph[u][n_u].get('weight', 1.0)
                if cur_bits[u] == cur_bits[n_u]: delta += w
                else: delta -= w
                    
            # Change in cut for neighbors of v (excluding u)
            for n_v in graph.neighbors(v):
                if n_v == u: continue
                w = graph[v][n_v].get('weight', 1.0)
                if cur_bits[v] == cur_bits[n_v]: delta += w
                else: delta -= w
                    
            if delta > 0:
                cur_bits[u] = 1 - cur_bits[u]
                cur_bits[v] = 1 - cur_bits[v]
                current_cut += delta
                improvement_found = True
                break # Break to restart the 1-bit sweep with the new landscape

print(f"Post-processing complete after {search_iteration} iteration.")
print(f"Final Deep-Optimized Cut Size: {current_cut}")


Initial Quantum Cut (Before Local Search): 6561.394409484451
Post-processing complete after 12 iteration.
Final Deep-Optimized Cut Size: 6853.430038477915
